## Setup and data profiling

In [1]:
import duckdb
import pandas as pd
from ydata_profiling import ProfileReport
import os

os.makedirs("DataProfile", exist_ok=True)

# 1. Load the dataset using DuckDB
file_path_tmp = os.path.join("data", "Combined", "resale_flat_prices_2012_01_to_2016_12.parquet")
file_path = file_path_tmp.replace('\\', '/')


master = duckdb.sql(f"SELECT * FROM '{file_path}'").df()

# --- Automated HTML Profiling Dashboard ---
print("Generating automated profiling HTML report...")
profile = ProfileReport(master, title="HDB Resale Flat Data Profiling (2012-2016)", progress_bar=False)

html_out = os.path.join("DataProfile", "HDB_Profiling_Report.html")
profile.to_file(html_out)
print(f"Saved automated report to: {html_out}\n")


# --- Profiling Metrics ---

print("==== Structure and Completeness ===")
structure_data = []
for col in master.columns:
    col_series = master[col]
    structure_data.append({
        "column": col,
        "data_type": str(col_series.dtype),
        "total_missing": int(col_series.isna().sum()),
        "unique_values": int(col_series.nunique())
    })
display(pd.DataFrame(structure_data))

print("\n=== Numeric Distribution ===")
numeric_cols = ["floor_area_sqm", "resale_price", "lease_commence_date", "remaining_lease"]
numeric_rows = []

for col in numeric_cols:
    if col in master.columns:
        s = pd.to_numeric(master[col], errors="coerce").dropna()
        if not s.empty:
            numeric_rows.append({
                "column": col,
                "count": len(s),
                "min": s.min(),
                "median": s.median(),
                "mean": round(s.mean(), 2),
                "p99": s.quantile(0.99),
                "max": s.max(),
                "std": round(s.std(), 2)
            })

display(pd.DataFrame(numeric_rows))


print("\n=== Categorical Vocabulary Analysis ===")
categorical_cols = ["town", "flat_type", "flat_model"]

for col in categorical_cols:
    if col in master.columns:
        print(f"\nTop values for: {col} (Total unique: {master[col].nunique()})")
        display(master[col].value_counts().head(5).rename("count").to_frame())


print("\n=== Storey Range Evolution (Months Active per Scheme) ===")
if "storey_range" in master.columns and "month" in master.columns:
    scheme_evolution = (
        master.groupby("storey_range")["month"]
        .agg(
            unique_months_count=lambda s: s.nunique(),
            months_active=lambda s: sorted(s.unique().tolist())
        )
        .reset_index()
        .sort_values("storey_range")
    )
    display(scheme_evolution)

C:\Keerthi\Carrier\HDB\Interview\Code\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\keert\AppData\Local\Temp\ipykernel_8412\4247177998.py:3: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Generating automated profiling HTML report...


  0%|          | 0/11 [00:00<?, ?it/s]

  9%|▉         | 1/11 [00:01<00:13,  1.32s/it]

...

,count
flat_type,
4 ROOM,36535
3 ROOM,26307
5 ROOM,21368
EXECUTIVE,7295
2 ROOM,956



Top values for: flat_model (Total unique: 20)


,count
flat_model,
Model A,26447
Improved,24117
New Generation,16495
Premium Apartment,8314
Simplified,5152



=== Storey Range Evolution (Months Active per Scheme) ===


,storey_range,unique_months_count,months_active
0,01 TO 03,57,"[2012-01, 2012-02, 2012-06, 2012-07, 2012-08, ..."
1,01 TO 05,3,"[2012-03, 2012-04, 2012-05]"
2,04 TO 06,57,"[2012-01, 2012-02, 2012-06, 2012-07, 2012-08, ..."
3,06 TO 10,3,"[2012-03, 2012-04, 2012-05]"
4,07 TO 09,57,"[2012-01, 2012-02, 2012-06, 2012-07, 2012-08, ..."
5,10 TO 12,57,"[2012-01, 2012-02, 2012-06, 2012-07, 2012-08, ..."
6,11 TO 15,3,"[2012-03, 2012-04, 2012-05]"
7,13 TO 15,57,"[2012-01, 2012-02, 2012-06, 2012-07, 2012-08, ..."
8,16 TO 18,57,"[2012-01, 2012-02, 2012-06, 2012-07, 2012-08, ..."
9,16 TO 20,3,"[2012-03, 2012-04, 2012-05]"


## Additional Checks if didnt satisfy 2012 Jan baseline

In [2]:
import numpy as np
import re

# 1.Load Data
file_path = os.path.join("data", "Combined", "resale_flat_prices_2012_01_to_2016_12.parquet")
df = duckdb.sql(f"SELECT * FROM '{file_path}'").df()
original_cols = list(df.columns)

# 2. Add Composite Key & Flag Duplicates
key_cols = [col for col in original_cols if col != 'resale_price']
df['composite_key'] = df[key_cols].astype(str).agg('|'.join, axis=1)

#  Flag Duplicates
df = df.sort_values(by='resale_price', ascending=False)
df['is_duplicate'] = df.duplicated(subset='composite_key', keep='first')

#  Jan 2012 Baseline Checks
jan_2012 = df[df['month'] == '2012-01']
valid_towns_2012 = set(jan_2012['town'].dropna())
valid_flat_types_2012 = set(jan_2012['flat_type'].dropna())
valid_flat_models_2012 = set(jan_2012['flat_model'].dropna())
valid_storey_ranges_2012 = set(jan_2012['storey_range'].dropna())

is_q1 = pd.DataFrame({
    'date': ~df['month'].astype(str).str.match(r'^\d{4}-\d{2}$'),
    'town': ~df['town'].isin(valid_towns_2012),
    'type': ~df['flat_type'].isin(valid_flat_types_2012),
    'model': ~df['flat_model'].isin(valid_flat_models_2012),
    'storey': ~df['storey_range'].isin(valid_storey_ranges_2012)
}).any(axis=1)

# Additional checks if data failed the Jan 2012 baseline. Could be new towns and flat models
official_towns = {t.upper() for t in [
    "Ang Mo Kio", "Bedok", "Bishan", "Bukit Batok", "Bukit Merah", 
    "Bukit Panjang", "Bukit Timah", "Central Area", "Choa Chu Kang", 
    "Clementi", "Geylang", "Hougang", "Jurong East", "Jurong West", 
    "Kallang/Whampoa", "Marine Parade", "Pasir Ris", "Punggol", 
    "Queenstown", "Sembawang", "Sengkang", "Serangoon", "Tampines", 
    "Tengah", "Toa Payoh", "Woodlands", "Yishun"
]}

valid_models = {m.upper() for m in [
    "Improved", "New Generation", "DBSS", "Standard", "Apartment", 
    "Simplified", "Model A", "Premium Apartment", "Adjoined flat", 
    "Model A-Maisonette", "Maisonette", "Type S1", "Type S2", 
    "Model A2", "Terrace", "Improved-Maisonette", "Premium Maisonette", 
    "Multi Generation", "Premium Apartment Loft", "2-room", "3Gen"
]}

def check_storey_violation(val):
    match = re.search(r"TO\s+(\d+)", str(val))
    return int(match.group(1)) >= 50 if match else False

is_q2 = pd.DataFrame({
    'town': ~df['town'].str.upper().isin(official_towns),
    'model': ~df['flat_model'].str.upper().isin(valid_models),
    'storey': df['storey_range'].apply(check_storey_violation)
}).any(axis=1)

#  Add reason for quarantine
conditions = [
    df['is_duplicate'],
    ~is_q1,
    is_q1 & ~is_q2,
    is_q1 & is_q2
]

choices = [
    "Duplicate Record (Lower Price)",
    "Valid (Passes Jan 2012 Baseline)",
    "Town, Flat Model, storey violates against 2012 but matched against valid list",
    "Town, Flat Model, storey violates against 2012 and failed refined validation checks"
]

df['data_quality_comment'] = np.select(conditions, choices, default="Unknown")

# 6. Split Data
valid_status = [
    "Valid (Passes Jan 2012 Baseline)",
    "Town, Flat Model, storey violates against 2012 but matched against valid list"
]

clean_df = df[df['data_quality_comment'].isin(valid_status)].copy()
quarantine_df = df[~df['data_quality_comment'].isin(valid_status)].copy()

# 7. Compute Remaining Lease for Clean Records
tx_year = clean_df['month'].str[:4].astype(int)
tx_month = clean_df['month'].str[5:7].astype(int)
commence_year = clean_df['lease_commence_date'].astype(int)

consumed_months = ((tx_year - commence_year) * 12) + (tx_month - 1)
remaining_total_months = (99 * 12) - consumed_months

rem_years = remaining_total_months // 12
rem_months = remaining_total_months % 12

clean_df['remaining_lease'] = rem_years.astype(str) + " years " + rem_months.astype(str).str.zfill(2) + " months"

# Add remaining_lease and composite_key back
clean_export_cols = original_cols.copy()
if 'remaining_lease' not in clean_export_cols:
    clean_export_cols.append('remaining_lease')
clean_export_cols.append('composite_key') 

quarantine_export_cols = original_cols + ['composite_key', 'data_quality_comment']
quarantine_export = quarantine_df[quarantine_export_cols]

print(f"Clean Records: {len(clean_df):,}")
print(f"Quarantined Records: {len(quarantine_export):,}")

# computed lease and key
print("\n===== Sample - Clean Output Preview (Key & Lease) ===")
display(clean_df.head(5))


print("\n===== Sample - Quarantined Records ===")
display(quarantine_export.head(5))

# Ensure directories exist before saving
os.makedirs(os.path.join("data", "Quarantined"), exist_ok=True)
os.makedirs(os.path.join("data", "Cleaned"), exist_ok=True)

quarantine_export.to_parquet(os.path.join("data", "Quarantined", "quarantined_records.parquet"), index=False)




Clean Records: 90,947
Quarantined Records: 1,597

===== Sample - Clean Output Preview (Key & Lease) ===


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,composite_key,is_duplicate,data_quality_comment
91881,2016-12,KALLANG/WHAMPOA,3 ROOM,57,JLN MA'MOR,01 TO 03,259.0,Terrace,1972,1150000.0,54 years 01 months,2016-12|KALLANG/WHAMPOA|3 ROOM|57|JLN MA'MOR|0...,False,Valid (Passes Jan 2012 Baseline)
86699,2016-09,CENTRAL AREA,5 ROOM,1G,CANTONMENT RD,43 TO 45,106.0,Type S2,2011,1120000.0,93 years 04 months,2016-09|CENTRAL AREA|5 ROOM|1G|CANTONMENT RD|4...,False,"Town, Flat Model, storey violates against 2012..."
89989,2016-11,CENTRAL AREA,5 ROOM,1D,CANTONMENT RD,46 TO 48,107.0,Type S2,2011,1100000.0,93 years 02 months,2016-11|CENTRAL AREA|5 ROOM|1D|CANTONMENT RD|4...,False,"Town, Flat Model, storey violates against 2012..."
85402,2016-08,KALLANG/WHAMPOA,5 ROOM,8,BOON KENG RD,28 TO 30,119.0,DBSS,2011,1100000.0,93 years 05 months,2016-08|KALLANG/WHAMPOA|5 ROOM|8|BOON KENG RD|...,False,"Town, Flat Model, storey violates against 2012..."
89988,2016-11,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,31 TO 33,106.0,Type S2,2011,1100000.0,93 years 02 months,2016-11|CENTRAL AREA|5 ROOM|1B|CANTONMENT RD|3...,False,"Town, Flat Model, storey violates against 2012..."



===== Sample - Quarantined Records ===


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,composite_key,data_quality_comment
31982,2013-07,BUKIT TIMAH,EXECUTIVE,7,TOH YI DR,04 TO 06,146.0,Maisonette,1989,950000.0,None,2013-07|BUKIT TIMAH|EXECUTIVE|7|TOH YI DR|04 T...,Duplicate Record (Lower Price)
74780,2016-02,CENTRAL AREA,4 ROOM,1B,CANTONMENT RD,49 TO 51,94.0,Type S1,2011,938880.0,93,2016-02|CENTRAL AREA|4 ROOM|1B|CANTONMENT RD|4...,"Town, Flat Model, storey violates against 2012..."
70687,2015-11,CENTRAL AREA,4 ROOM,1A,CANTONMENT RD,37 TO 39,95.0,Type S1,2011,915000.0,94,2015-11|CENTRAL AREA|4 ROOM|1A|CANTONMENT RD|3...,Duplicate Record (Lower Price)
64446,2015-07,CENTRAL AREA,4 ROOM,1G,CANTONMENT RD,49 TO 51,94.0,Type S1,2011,910000.0,94,2015-07|CENTRAL AREA|4 ROOM|1G|CANTONMENT RD|4...,"Town, Flat Model, storey violates against 2012..."
32888,2013-07,TOA PAYOH,5 ROOM,81,LOR 4 TOA PAYOH,19 TO 21,122.0,Improved,1997,880000.0,None,2013-07|TOA PAYOH|5 ROOM|81|LOR 4 TOA PAYOH|19...,Duplicate Record (Lower Price)


## Anomaly Detection using The interquartile range (IQR)

In [3]:
# IQR Anomaly Detection on clean_df
iqr_query = """
WITH base_data AS (
    SELECT 
        *,
        (resale_price / floor_area_sqm) AS psm,
        CAST(month || '-01' AS DATE) AS tx_date,
        FLOOR((99 - (CAST(SUBSTR(month, 1, 4) AS INT) - lease_commence_date)) / 5) * 5 AS lease_bin_5yr
    FROM clean_df
),
windowed_stats AS (
    SELECT 
        *,
        COUNT(psm) OVER w_street AS cohort_count,
        quantile_cont(psm, 0.25) OVER w_street AS q1,
        quantile_cont(psm, 0.75) OVER w_street AS q3
    FROM base_data
    WINDOW 
        w_street AS (
            PARTITION BY street_name, storey_range, flat_type, lease_bin_5yr 
            ORDER BY tx_date 
            RANGE BETWEEN INTERVAL 1 YEAR PRECEDING AND INTERVAL 1 YEAR FOLLOWING
        )
),
scored_data AS (
    SELECT 
        *,
        (q3 - q1) AS iqr,
        (q1 - (1.5 * (q3 - q1))) AS lower_bound,
        (q3 + (1.5 * (q3 - q1))) AS upper_bound,
        CASE 
            WHEN cohort_count >= 4 THEN 'Street Level (+/- 1 Year)'
            ELSE 'Insufficient Data' 
        END AS baseline_used
    FROM windowed_stats
)
SELECT 
    *,
    CASE 
        WHEN baseline_used = 'Insufficient Data' THEN 'Insufficient Data'
        WHEN psm > upper_bound THEN 'Anomaly - Higher'
        WHEN psm < lower_bound THEN 'Anomaly - Lower'
        ELSE 'Normal'
    END AS anomaly_status
FROM scored_data
"""

clean_df = duckdb.sql(iqr_query).df()

# Include data_quality_comment --> Insufficient Data (OR) Normal (OR) Anomaly - Higher (OR) Anomaly - Lower
clean_export_cols = list(dict.fromkeys(
    original_cols + ['composite_key', 'data_quality_comment', 'remaining_lease', 'psm', 'baseline_used', 'lower_bound', 'upper_bound', 'anomaly_status']
))
quarantine_export_cols = list(dict.fromkeys(original_cols + ['composite_key', 'data_quality_comment']))

quarantine_export = quarantine_df[quarantine_export_cols]
clean_export = clean_df[clean_export_cols]

print(f"Clean Records Exported      : {len(clean_export):,}")
print(f"Quarantined Records Exported: {len(quarantine_export):,}")
print("\n=== Clean Records Anomaly Breakdown ===")
print(clean_export['anomaly_status'].value_counts())

import os

# Ensure directories exist
os.makedirs(os.path.join("data", "Quarantined"), exist_ok=True)
os.makedirs(os.path.join("data", "Cleaned"), exist_ok=True)

quarantine_export.to_parquet(os.path.join("data", "Quarantined", "quarantined_records.parquet"), index=False)
clean_export.to_parquet(os.path.join("data", "Cleaned", "cleaned_records.parquet"), index=False)

Clean Records Exported      : 90,947
Quarantined Records Exported: 1,597

=== Clean Records Anomaly Breakdown ===
anomaly_status
Normal               59679
Insufficient Data    28752
Anomaly - Higher      1302
Anomaly - Lower       1214
Name: count, dtype: int64
